# 01 — PreFlight: Data Intake & Quality Control


In [ ]:
import sys
from pathlib import Path
REPO = Path.cwd()
for candidate in [REPO, REPO.parent, REPO.parent.parent]:
    if (candidate / "src" / "neuro").exists():
        REPO = candidate
        break
sys.path.insert(0, str(REPO / "src"))
import os
os.chdir(REPO)
%matplotlib inline
import matplotlib.pyplot as plt
import seaborn as sns
sns.set_theme(style="whitegrid", context="notebook")


In [ ]:

import json, psutil
import pandas as pd
import nibabel as nib
import mlflow
from neuro.config import DATA_ROOT, TR_SEC
from neuro.bids import validate_bids, group_counts
from neuro.spark_utils import get_spark, participants_spark
from neuro.mlflow_utils import start_run
from neuro import viz

with start_run("01_pre_flight"):
    report = validate_bids()
    participants = report["participants"]
    runs = report["runs"]
    mlflow.log_param("n_subjects", report["n_subjects"])
    mlflow.log_param("tr_sec", report["tr_json_music"])
    print("=== BIDS Pre-flight ===")
    for k, v in report.items():
        if k not in ("missing", "runs", "participants"):
            print(f"{k}: {v}")
    print("\nGroup counts:")
    display(group_counts())
    print(f"\nMemory available: {psutil.virtual_memory().available / 1e9:.1f} GB")


In [ ]:

spark = get_spark("BladerunnerNeuro_PreFlight")
spark.sql("SELECT 'Spark ready for scaled preprocessing' AS status").show()
participants_spark(spark, participants).groupBy("group_short").count().show()


In [ ]:

if report["n_runs_available"]:
    sample = runs[runs["bold_exists"]].iloc[0]
    img = nib.load(sample["bold_path"])
    print("Sample:", sample["bold_path"])
    print("Shape:", img.shape, "Zooms:", img.header.get_zooms())
viz.plot_group_demographics(participants)
plt.show()
viz.plot_tr_histogram(runs)
plt.show()
report["missing"].head(10)
